In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer
import os
import random
import h5py

In [2]:
import random
import numpy as np
import tensorflow as tf

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)


In [3]:
# Hyperparameters
window_size = 500               # 500 time steps (~0.25 seconds at 2034 Hz)
stride = 250                    # 50% overlap
downsample_factor = 20           
# batch_size = 32
# epochs = 20


In [4]:

def get_dataset_name(file_name_with_dir):
    
    filename_without_dir = file_name_with_dir.split('/')[-1]
    print(filename_without_dir)
    temp = filename_without_dir.split('_')[:-1]
    print(temp)
    dataset_name = "_".join(temp)
    return dataset_name
filename_path="./data/Intra/train/rest_105923_1.h5"
with h5py. File (filename_path , 'r') as f :
    dataset_name = get_dataset_name(filename_path)
    print(dataset_name)
    matrix = f.get(dataset_name)[()]
    print(type(matrix ))
    print(matrix.shape)

rest_105923_1.h5
['rest', '105923']
rest_105923
<class 'numpy.ndarray'>
(248, 35624)


In [5]:
def z_score_normalization(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return (data - mean) / std

In [6]:
def segment_data(data, label, window_size=500, stride=500):
    segments = []
    labels = []
    for start in range(0, data.shape[1] - window_size + 1, stride):
        end = start + window_size
        segment = data[:, start:end]
        segments.append(segment)
        labels.append(label)
    return segments, labels

In [7]:
def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")

In [8]:
def load_and_preprocess(filepath, label_map, window_size=500, stride=500, downsample_factor=20):
    filename = filepath.lower()
    task_label = infer_label_from_filename(filepath, label_map)
    # Load data
    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # Shape: (248, 35624)

    
# Segment the data into overlapping windows
    segments = []
    labels = []
    num_timepoints = data.shape[1]

    for start in range(0, num_timepoints - window_size + 1, stride):
        end = start + window_size
        window = data[:, start:end]

        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # Add channel dim for CNN
        labels.append(task_label)

    
    return segments, labels

In [9]:
def data_generator(filepaths, label_map, batch_size=8):
    while True:  # Infinite generator
        all_segments, all_labels = [], []
        for filepath in filepaths:
            segments, labels = load_and_preprocess(filepath, label_map)
            all_segments.extend(segments)
            all_labels.extend(labels)

        X = np.array(all_segments)
        y = np.array(all_labels)

        indices = np.arange(len(y))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

        for i in range(0, len(X), batch_size):
            yield X[i:i+batch_size], y[i:i+batch_size]

In [10]:
# Parameters
train_dir = './data/Cross/train'
batch_size = 4  # number of files per iteration
epochs_per_batch = 1  # how many epochs for each file batch
total_epochs = 10  # total desired training epochs
l2_lambda = 0.001  # regularization strength

# Load all file paths and shuffle
all_filepaths = [
    os.path.normpath(os.path.join(train_dir, fname))
    for fname in os.listdir(train_dir)
    if fname.endswith('.h5')
]
np.random.shuffle(all_filepaths)

# Label map
label_map = {
    'rest': 0,
    'math': 1,
    'story': 1,
    'story_math': 1,         
    'working_memory': 2,
    'memory': 2,
    'motor': 3
}


In [11]:
segments, _ = load_and_preprocess(all_filepaths[0], label_map)
print(f"Segment shape: {segments[0].shape}")

Segment shape: (248, 500, 1)


In [12]:
from keras import layers, models, regularizers
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
def build_cnn(input_shape=segments[0].shape, num_classes=4, l2_lambda=0.01): # shape = (248, 35, 1)
    model = Sequential([
        InputLayer(input_shape=input_shape),

        Conv2D(32, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),
 

        Conv2D(64, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Conv2D(128, (3, 3), activation='relu', padding='same',
               kernel_regularizer=l2(l2_lambda)),
        BatchNormalization(),
        MaxPooling2D((1, 2)),


        Flatten(),
       #  Dropout(0.6),
        Dense(128, activation='relu', kernel_regularizer=l2(l2_lambda)),
        Dropout(0.7),
        Dense(num_classes, activation='softmax')
    ])
    
   
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [13]:
def load_test_data(test_filepaths, label_map, window_size=500, stride=500, downsample_factor=20):
    all_segments = []
    all_labels = []

    for filepath in test_filepaths:
        segments, labels = load_and_preprocess(
            filepath, label_map,
            window_size=window_size,
            stride=stride,
            downsample_factor=downsample_factor
        )
        all_segments.extend(segments)
        all_labels.extend(labels)

    X_test = np.array(all_segments)
    y_test = np.array(all_labels)
    return X_test, y_test


In [14]:
import os
import numpy as np
from sklearn.utils import shuffle

def chunked_training(model, train_filepaths, label_map, chunk_size=4, epochs=5, batch_size=8):
    num_chunks = len(train_filepaths) // chunk_size

    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} ===")
        shuffled_files = shuffle(train_filepaths)

        for i in range(num_chunks):
            chunk_files = shuffled_files[i * chunk_size : (i + 1) * chunk_size]
            print(f"\nTraining on files {i * chunk_size + 1} to {(i + 1) * chunk_size}")

            # Load and preprocess only this chunk
            X_batch, y_batch = load_test_data(chunk_files, label_map)

            # Fit model on this chunk
            model.fit(X_batch, y_batch, batch_size=batch_size, epochs=1, verbose=1)

    return model


In [15]:
def count_total_segments(filepaths, label_map):
    total = 0
    for fp in filepaths:
        segments, _ = load_and_preprocess(fp, label_map)
        total += len(segments)
    return total

In [16]:
from sklearn.model_selection import train_test_split
from glob import glob

train_folder = './data/Cross/train'
train_filepaths = glob(os.path.join(train_folder, '*.h5'))
train_filepaths = [os.path.normpath(p) for p in train_filepaths]

train_files, val_files = train_test_split(train_filepaths, test_size=0.2, random_state=42)
train_gen = data_generator(train_files, label_map, batch_size )
val_gen   = data_generator(val_files, label_map, batch_size)
# Estimate how many segments per file
train_total_segments = count_total_segments(train_filepaths, label_map)
steps_per_epoch = train_total_segments // batch_size
val_total_segments = count_total_segments(val_files, label_map)
val_steps_per_epoch = val_total_segments // batch_size


In [18]:
from glob import glob

from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model = build_cnn(
)

model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=200,
    # steps_per_epoch,
    validation_steps=200,
    # val_steps_per_epoch,
    epochs=5,
    callbacks=[early_stop],
    verbose=1
)


d:\projects\DL-Assignment-2\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 434s 2s/step - accuracy: 0.3543 - loss: 5.1187 - val_accuracy: 0.1600 - val_loss: 4.3627
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 408s 2s/step - accuracy: 0.6486 - loss: 3.6217 - val_accuracy: 0.1502 - val_loss: 5.3349
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.6834 - loss: 3.1121 - val_accuracy: 0.2466 - val_loss: 17.1448
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 409s 2s/step - accuracy: 0.8422 - loss: 2.4059 - val_accuracy: 0.3066 - val_loss: 309.4538


In [19]:
import glob

for i in range(1, 4):
    # Collect test files
    test_folder = f"./data/Cross/test{i}"
    test_filepaths = glob.glob(os.path.join(test_folder, '*.h5'))
    test_filepaths = [os.path.normpath(p) for p in test_filepaths]

    # Load and preprocess test data
    X_test, y_test = load_test_data(test_filepaths, label_map)
    # Option A: Direct evaluation
    loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
    print(f"Test Accuracy: {accuracy}")

36/36 ━━━━━━━━━━━━━━━━━━━━ 28s 772ms/step - accuracy: 0.2533 - loss: 4.3582
Test Accuracy: 0.25
36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.2533 - loss: 4.3582
Test Accuracy: 0.25
36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.2533 - loss: 4.3582
Test Accuracy: 0.25
